# 09 · Verificar y devolver resultados

Usa CPU. No entrena ni evalúa test. Verifica las 120 corridas principales esperadas, incluyendo las históricas. Los resultados precalculados o archivos ausentes se reportan; no se inventan.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
from pathlib import Path
import json, hashlib, zipfile, subprocess, sys
CAMPANA = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4')
BENCHMARK = Path('/content/drive/MyDrive/tesis_imo_e2_arquitecturas_reales_v2')
COMPACTAS = Path('/content/compactas_e3_recibidas')
ZIP = Path('/content/drive/MyDrive/compactas_a0498_e3.zip')
assert BENCHMARK.is_dir(), 'Falta la carpeta histórica que confirmaste en Drive'
assert ZIP.is_file(), 'Sube compactas_a0498_e3.zip a MyDrive'
COMPACTAS.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    for name in z.namelist():
        assert (COMPACTAS/name).resolve().is_relative_to(COMPACTAS)
    z.extractall(COMPACTAS)
ESPERADOS = {'fcn_a0498_s11': 'ef655608d580e34a72966f6f352e643544c3eb8c215dace5d246da98f72814c7', 'fcn_a0498_s23': 'c9f9856e9239d9c59366871525b870ac421ef6ae370bb60b29bff6960aeb2772', 'fcn_a0498_s37': 'f5c326a78e22f4d8dca84cdfd0b86a9cd9646e947ab22c59465fd82d4af00df4', 'fcn_a0498_s53': '6f220654d54344f5b6545087d3c102c0415be2d97eed8721b62ebb36789a86fa', 'fcn_a0498_s71': '9e2b5d911b7722573885eeb157cf75f43b5302d11eabcb1ad287f6c16ff1164e', 'segnet_a0498_s11': 'c2bc57be34caa5501118601b9d050be50062f5a657309850e47e2a96b5ac2d14', 'segnet_a0498_s23': 'a5d63031449a429343961f982ef26fdcf6817c139fb1c7142a3b0fd4199d2dac', 'segnet_a0498_s37': '7c358e59dd5ef539375b43d78a30b5f46de262418afac998a956bea0bc75e750', 'segnet_a0498_s53': '3c41962731ca6d1680a879f6d25608abcda7790c6e24e91ed68835463bc1376c', 'segnet_a0498_s71': 'c18f29d092e404dc42210bf284fb4b998c02eebda495322ac42e9eb2c8518ecd', 'unet_a0498_s11': '4138b06d20bd32a422a69970252187322a35fea09a4e304bcda61d904ac672a9', 'unet_a0498_s23': 'e0452e0bf9df896ffaafbbc1838c02abc070d1b37478121fc25246a33aec75aa', 'unet_a0498_s37': '59c6df9ad6c05500114d0e5e4948e8fd4cc5bd831eda4c577aab6066be25ddeb', 'unet_a0498_s53': '615d2b1a786ec37372e8ca2868783ec99ce135c556b63610b9a7bfc1e4a10f02', 'unet_a0498_s71': 'a0d80416558ccdfd1632ad34ef1bd132ac896f2c83ab7b8b7c2f5d0adb03df84'}
for run, expected in ESPERADOS.items():
    assert hashlib.sha256((COMPACTAS/'fase2'/run/'best.pt').read_bytes()).hexdigest()==expected


In [ ]:
SCRIPT = '"""Inventario por rutas explícitas, sin inferir carpetas ni cargar imágenes de test."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\n\nSEEDS = [11, 23, 37, 53, 71, 89, 97, 101, 103, 107]\nOTHERS = [\'transunet_2d\', \'pidnet_s\', \'vmunet\', \'segnext_t\', \'deeplabv3plus\', \'unetplusplus\']\n\n\ndef sha(p):\n    h = hashlib.sha256()\n    with p.open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--campana\', type=Path, required=True)\n    ap.add_argument(\'--benchmark\', type=Path, required=True)\n    ap.add_argument(\'--compactas\', type=Path, required=True)\n    args = ap.parse_args()\n    targets = []\n    for s in SEEDS:\n        for arm in [\'dbnet\', \'dbnet_at\']:\n            targets.append((f\'unetat_{arm}\', s, args.campana / \'original\' / f\'unetat_{arm}_seed{s}\'))\n        for family in [\'fcn\', \'unet\', \'segnet\']:\n            root = args.compactas if s in SEEDS[:5] else args.campana / \'compactas\'\n            targets.append((family, s, root / \'fase2\' / f\'{family}_a0498_s{s}\'))\n        for model in OTHERS + [\'segformer_b0\']:\n            session = \'A\' if model in OTHERS[:2] else \'B\'\n            if model == \'segformer_b0\': session = \'C\'\n            root = args.benchmark if s in SEEDS[:5] and model != \'segformer_b0\' else args.campana / f\'rivales/sesion_{session}\'\n            targets.append((model, s, root / f\'{model}_seed{s}\'))\n    rows = []\n    for model, seed, directory in targets:\n        row = dict(modelo=model, semilla=seed, directorio=str(directory), estado=\'falta_resultado\', dice_val=\'\')\n        rp = directory / \'result.json\'\n        if rp.exists():\n            try:\n                r = json.loads(rp.read_text())\n                cfg = json.loads((directory / \'config.json\').read_text())\n                assert cfg[\'fingerprint\'] == r[\'fingerprint\']\n                assert r.get(\'completed\') is True and r.get(\'seed\', r.get(\'semilla\')) == seed\n                assert r.get(\'model\', r.get(\'familia\')) == model\n                if model in [\'fcn\', \'unet\', \'segnet\']:\n                    assert r[\'arq\'] == \'a0498\'\n                    assert cfg[\'tarea\'][\'w\'] == [64,64,64] and cfg[\'tarea\'][\'b\'] == [0,1,0]\n                if model.startswith(\'unetat_\'):\n                    assert cfg[\'identity\'][\'opencv\'] == \'4.11.0\'\n                if model == \'segformer_b0\':\n                    recipe = cfg[\'config\'][\'models\'][model]\n                    assert recipe[\'optimizer\']==\'AdamW\' and abs(recipe[\'lr\']-.0005573638936676344)<1e-14\n                    assert recipe[\'weight_decay\']==.0001\n                name = r[\'best_file\']\n                assert Path(name).name == name\n                assert sha(directory / name) == r[\'best_sha256\']\n                row.update(estado=\'verificado\', dice_val=r[\'best_val_group_Dice\'])\n            except (KeyError, ValueError, AssertionError, OSError) as e:\n                row[\'estado\'] = \'invalido:\' + type(e).__name__\n        rows.append(row)\n    args.campana.mkdir(parents=True, exist_ok=True)\n    with (args.campana / \'inventario_campana.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\n        writer.writeheader(); writer.writerows(rows)\n    report = dict(esperadas=len(rows), verificadas=sum(r[\'estado\']==\'verificado\' for r in rows),\n                  test_evaluado=False, pendientes=[r for r in rows if r[\'estado\']!=\'verificado\'])\n    report[\'lista_para_revision_pretest\'] = report[\'verificadas\'] == report[\'esperadas\']\n    (args.campana / \'estado_campana.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n    print(json.dumps(report, indent=2))\n    return 0 if report[\'lista_para_revision_pretest\'] else 2\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n'
p = Path('/content/auditar_campana.py'); p.write_text(SCRIPT, encoding='utf-8')
r = subprocess.run([sys.executable,str(p),'--campana',str(CAMPANA),
    '--benchmark',str(BENCHMARK),'--compactas',str(COMPACTAS)])
assert r.returncode in [0,2], 'Error del auditor'
print((CAMPANA/'estado_campana.json').read_text())
archive = Path('/content/resultados_e3_secuencia_v4.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in CAMPANA.rglob('*'):
        if p.is_file() and p.suffix in ['.json','.csv','.txt']:
            z.write(p,str(p.relative_to(CAMPANA)))
files.download(str(archive))
print('Los pesos permanecen en Drive. Devuelve este ZIP para integración y revisión antes de test.')
